# Per-Category Activation Steering Configuration Finder v3.0

Train dedicated steering vectors for each harm category with **per-category layer stability analysis**.

## Key Features
- **Per-category layer stability**: Each category finds its own optimal layers
- **Configurable presets**: `quick` (~30 min) vs `full` (~70 min)
- **Comprehensive data capture**: All results saved to YAML for future analysis
- **Clean progress reporting**: No library spam, clear ETAs

## Categories
violence, cybercrime, fraud, drugs, hate, sexual

## Outputs
```
steering_config/
├── run_metadata.yaml          # Run timing, system info
├── config.yaml                # Master configuration (for 10b-apply)
└── {category}/
    ├── layer_stability.yaml   # ALL layer coherence data
    ├── condition_search.yaml  # ALL condition point search results
    ├── range_optimization.yaml # ALL strength test results
    └── *.svec                 # Steering vectors
```

In [48]:
# Cell 1: Imports
import torch
import numpy as np
import yaml
import re
import gc
import os
import sys
import math
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any, Union
from collections import Counter
from dataclasses import dataclass, field, asdict
from contextlib import contextmanager
from io import StringIO
from tqdm.auto import tqdm

# ML/AI imports
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from activation_steering import SteeringVector, SteeringDataset, MalleableModel
from activation_steering.config import GlobalConfig

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu126 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


PyTorch: 2.9.1+cu126
CUDA: True
GPU: NVIDIA GeForce RTX 4080 SUPER
Memory: 15.6 GB


In [49]:
# Cell 2: GPU Utilities + Output Suppression
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@contextmanager
def suppress_output():
    """Suppress both stdout and stderr to silence all library output."""
    old_stdout, old_stderr = sys.stdout, sys.stderr
    devnull = open(os.devnull, 'w')
    sys.stdout, sys.stderr = devnull, devnull
    try:
        yield
    finally:
        sys.stdout, sys.stderr = old_stdout, old_stderr
        devnull.close()

def print_gpu(label=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        free = total - torch.cuda.memory_reserved() / 1024**3
        print(f"[GPU {label}] {alloc:.2f}GB used | {free:.2f}GB free")

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

print(f"Device: {device}")
print_gpu("Initial")

Device: cuda
[GPU Initial] 0.00GB used | 15.57GB free


In [50]:
# Cell 3: Coherence Detection

def compute_ngram_diversity(text: str, n: int = 2) -> float:
    words = text.lower().split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    return len(set(ngrams)) / len(ngrams) if ngrams else 0.0

def compute_char_entropy(text: str) -> float:
    if not text:
        return 0.0
    counts = Counter(text.lower())
    total = sum(counts.values())
    return -sum((c/total) * math.log2(c/total) for c in counts.values() if c > 0)

def is_coherent_text(text: str) -> Tuple[bool, dict]:
    """5-metric coherence check."""
    if len(text.strip()) < 10:
        return False, {"reason": "too_short"}
    
    words = text.split()
    unique_ratio = len(set(w.lower() for w in words)) / len(words) if words else 0
    bigram_div = compute_ngram_diversity(text, 2)
    entropy = compute_char_entropy(text)
    
    # Consecutive repeats
    consec = sum(1 for i in range(1, len(words)) if words[i] == words[i-1])
    consec_ratio = consec / (len(words) - 1) if len(words) > 1 else 0
    
    score = 0
    if unique_ratio >= 0.2: score += 1
    if bigram_div >= 0.3: score += 1
    if entropy >= 3.0: score += 1
    if consec_ratio < 0.3: score += 1
    score += 1  # Skip cyclic check for speed
    
    return score >= 4, {
        "score": score, "unique_ratio": unique_ratio,
        "bigram_div": bigram_div, "entropy": entropy, "consec_ratio": consec_ratio
    }

print("[OK] Coherence detection")

[OK] Coherence detection


In [51]:
# Cell 4: Load YAML Config + Resolve Preset

NOTEBOOK_DIR = Path("/workspace/Sync/AI/bazzite/bazzite-ai-notebooks")
DATASET_PATH = NOTEBOOK_DIR / "steering_dataset.yaml"

with open(DATASET_PATH, "r") as f:
    dataset = yaml.safe_load(f)

# Get mode and preset
MODE = dataset.get("mode", "quick")
PRESET = dataset["presets"][MODE]

# Extract preset config
STABILITY_CFG = PRESET["stability"]
CONDITION_CFG = PRESET["condition"]
TRAINING_CFG = PRESET["training"]
RANGE_CFG = PRESET["range_optimization"]
GENERATION_CFG = PRESET["generation"]
OUTPUT_CFG = PRESET["output"]

# Extract data
behavior = dataset["behavior"]
questions = behavior["questions"]
refusal_suffixes = behavior["refusal"]
compliance_suffixes = behavior["compliance"]

condition = dataset["condition"]
CATEGORIES = list(condition["harmful"].keys())
harmful_by_category = condition["harmful"]
harmless_prompts = condition["harmless"]

tests = dataset["tests"]
test_harmful_by_category = tests["harmful"]
test_harmless = tests["harmless"]

REFUSAL_INDICATORS = dataset["indicators"]

def is_refusal(text: str) -> bool:
    return any(re.search(p, text.lower()) for p in REFUSAL_INDICATORS)

def get_test_prompts(category: str) -> List[Tuple[str, str]]:
    harmful = [(p, "harmful") for p in test_harmful_by_category.get(category, [])]
    harmless = [(p, "harmless") for p in test_harmless]
    return harmful + harmless

print(f"[OK] Config v{dataset['metadata']['version']}")
print(f"  Mode: {MODE}")
print(f"  Categories: {CATEGORIES}")
print(f"  Stability: {len(STABILITY_CFG['strengths'])} strengths, {STABILITY_CFG['test_prompts']} prompts, min_range={STABILITY_CFG['min_range_width']}")
print(f"  Condition: layers {CONDITION_CFG['layer_range']}, step {CONDITION_CFG['threshold_step']}")
print(f"  Training: {TRAINING_CFG['max_layers']} layers, blend {TRAINING_CFG['blend_ratio']}")
print(f"  Range: fwd={RANGE_CFG['forward_strengths']}, rev={RANGE_CFG['reverse_strengths']}")
print(f"  Generation: stability={GENERATION_CFG['max_new_tokens_stability']}, test={GENERATION_CFG['max_new_tokens_test']}")
print(f"  Output: save_responses={OUTPUT_CFG['save_responses']}")

[OK] Config v3.0
  Mode: quick
  Categories: ['violence', 'cybercrime', 'fraud', 'drugs', 'hate', 'sexual']
  Stability: 4 strengths, 1 prompts, min_range=1.0
  Condition: layers [8, 30], step 0.03
  Training: 4 layers, blend 0.7
  Range: fwd=[1.5, 3], rev=[-1.5, -3]
  Generation: stability=25, test=35
  Output: save_responses=True


In [52]:
# Cell 5: Load Model

MODEL_ID = dataset["metadata"]["model"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID}...")
print_gpu("Before")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_SIZE = model.config.hidden_size

print(f"\n[OK] {NUM_LAYERS} layers, {HIDDEN_SIZE} hidden")
print_gpu("After")

Loading mistralai/Ministral-8B-Instruct-2410...
[GPU Before] 0.00GB used | 15.57GB free


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.



[OK] 36 layers, 4096 hidden
[GPU After] 5.34GB used | 8.31GB free


In [53]:
# Cell 6: Core Functions + MCP Progress System

# =============================================================================
# Data Classes
# =============================================================================

@dataclass
class LayerStabilityData:
    """Complete stability data for one layer."""
    layer: int
    max_forward: float = 0.0
    max_reverse: float = 0.0
    range_width: float = 0.0
    stable: bool = False
    coherence_by_strength: Dict[float, float] = field(default_factory=dict)
    responses: Dict[float, List[dict]] = field(default_factory=dict)

@dataclass
class CategoryResult:
    """Complete results for one category."""
    category: str
    # Vectors
    forward: SteeringVector = None
    reverse: SteeringVector = None
    condition: SteeringVector = None
    blended: SteeringVector = None
    # Layer stability (per-category)
    layer_stability: Dict[int, LayerStabilityData] = field(default_factory=dict)
    stable_layers: List[int] = field(default_factory=list)
    training_layers: List[int] = field(default_factory=list)
    # Condition search
    condition_layer: List[int] = field(default_factory=list)
    condition_threshold: float = 0.0
    condition_direction: str = "positive"
    condition_f1: float = 0.0
    # Range optimization
    range_data: Dict = field(default_factory=dict)
    optimal_forward: float = 0.75
    max_forward: float = 1.0
    max_reverse: float = -1.0
    balanced_range: float = 0.75
    # Timing
    timing: Dict[str, float] = field(default_factory=dict)

# =============================================================================
# Per-Preset Progress Tracker with Accumulated Statistics
# =============================================================================

@dataclass
class ProgressTracker:
    """Progress tracking with per-preset files and accumulated statistics."""
    total_categories: int
    output_dir: Path
    mode: str = "quick"  # Preset name for filename
    start_time: float = field(default_factory=time.time)
    
    # Existing data from previous runs (loaded from preset file)
    existing_data: Dict = field(default_factory=dict)
    
    # Timing history for current run
    category_times: List[float] = field(default_factory=list)
    phase_times: Dict[str, List[float]] = field(default_factory=dict)
    
    # Detailed per-category timing history
    category_phase_history: Dict[str, Dict[str, float]] = field(default_factory=dict)
    
    # Current state
    current_category: int = 0
    current_category_name: str = ""
    current_phase: int = 0
    current_phase_name: str = ""
    current_detail: str = ""
    category_start: float = 0
    phase_start: float = 0
    
    # Phase names for reference
    PHASE_NAMES = {
        1: "behavior_vectors",
        2: "layer_stability", 
        3: "condition_vector",
        4: "condition_search",
        5: "range_optimization"
    }
    
    def __post_init__(self):
        """Load existing progress data for this preset."""
        self._load_existing_progress()
    
    def _load_existing_progress(self):
        """Load existing progress file for this preset."""
        progress_path = self.output_dir / f"progress_{self.mode}.yaml"
        if progress_path.exists():
            try:
                with open(progress_path) as f:
                    self.existing_data = yaml.safe_load(f) or {}
            except Exception:
                self.existing_data = {}
        else:
            self.existing_data = {}
    
    def start_category(self, name: str):
        """Begin processing a category."""
        self.current_category += 1
        self.current_category_name = name
        self.current_phase = 0
        self.current_phase_name = ""
        self.current_detail = ""
        self.category_start = time.time()
        self.category_phase_history[name] = {}
        self._update_progress_file()
        self._print_progress()
    
    def start_phase(self, phase_num: int, name: str):
        """Begin a phase within category."""
        self.current_phase = phase_num
        self.current_phase_name = name
        self.current_detail = ""
        self.phase_start = time.time()
        self._update_progress_file()
        self._print_progress()
    
    def update_detail(self, detail: str):
        """Update current operation detail."""
        self.current_detail = detail
        self._update_progress_file()
    
    def complete_phase(self):
        """Mark phase complete and record timing."""
        elapsed = time.time() - self.phase_start
        phase_key = self.PHASE_NAMES.get(self.current_phase, self.current_phase_name)
        
        # Record in phase_times list for averaging
        if phase_key not in self.phase_times:
            self.phase_times[phase_key] = []
        self.phase_times[phase_key].append(elapsed)
        
        # Record in category-specific history
        if self.current_category_name in self.category_phase_history:
            self.category_phase_history[self.current_category_name][phase_key] = elapsed
        
        self._update_progress_file()
    
    def complete_category(self):
        """Mark category complete and record timing."""
        elapsed = time.time() - self.category_start
        self.category_times.append(elapsed)
        
        # Store total in category history
        if self.current_category_name in self.category_phase_history:
            self.category_phase_history[self.current_category_name]["_total"] = elapsed
        
        self._update_progress_file()
        self._print_progress()
    
    def skip_category(self, name: str, reason: str = "already complete"):
        """Record a skipped category."""
        self.current_category += 1
        print(f"  [SKIP] {name} ({reason})")
    
    def estimate_remaining(self) -> str:
        """Estimate remaining time based on completed work."""
        if not self.category_times:
            return "calculating..."
        
        avg_category = sum(self.category_times) / len(self.category_times)
        remaining_categories = self.total_categories - self.current_category
        remaining_seconds = avg_category * remaining_categories
        
        # Add partial estimate for current category
        if self.current_phase > 0 and self.current_phase < 5:
            phases_remaining = 5 - self.current_phase
            for phase_name, times in self.phase_times.items():
                if times:
                    remaining_seconds += (sum(times) / len(times)) * (phases_remaining / 5)
        
        return self._format_time(remaining_seconds)
    
    def _format_time(self, seconds: float) -> str:
        """Format seconds as human-readable string."""
        if seconds < 60:
            return f"{seconds:.1f}s"
        elif seconds < 3600:
            mins = int(seconds // 60)
            secs = int(seconds % 60)
            return f"{mins}m {secs}s"
        else:
            hours = int(seconds // 3600)
            mins = int((seconds % 3600) // 60)
            return f"{hours}h {mins}m"
    
    def _merge_statistics(self) -> Dict:
        """Merge current run statistics with existing data from previous runs."""
        existing_stats = self.existing_data.get("statistics", {})
        existing_details = existing_stats.get("phase_details", {})
        
        merged_phase_details = {}
        for phase_name, times in self.phase_times.items():
            if not times:
                continue
            
            old = existing_details.get(phase_name, {})
            old_count = old.get("count", 0)
            old_total = old.get("total_seconds", 0)
            old_min = old.get("min_seconds")
            old_max = old.get("max_seconds", 0)
            old_recent = old.get("recent_times", [])
            
            new_count = old_count + len(times)
            new_total = old_total + sum(times)
            
            # Handle min (could be None from previous runs)
            current_min = min(times)
            if old_min is not None:
                new_min = min(current_min, old_min)
            else:
                new_min = current_min
            
            new_max = max(max(times), old_max)
            new_recent = (old_recent + [round(t, 2) for t in times])[-6:]  # Keep last 6
            
            merged_phase_details[phase_name] = {
                "count": new_count,
                "total_seconds": round(new_total, 2),
                "avg_seconds": round(new_total / new_count, 2) if new_count > 0 else 0,
                "min_seconds": round(new_min, 2),
                "max_seconds": round(new_max, 2),
                "recent_times": new_recent,
            }
        
        # Compute phase averages from merged details
        phase_averages = {
            name: details["avg_seconds"]
            for name, details in merged_phase_details.items()
        }
        
        # Append category totals (keep last 18)
        old_totals = existing_stats.get("category_totals", [])
        new_totals = (old_totals + [round(t, 2) for t in self.category_times])[-18:]
        
        return {
            "phase_averages": phase_averages,
            "phase_details": merged_phase_details,
            "category_totals": new_totals,
        }
    
    def _print_progress(self):
        """Print structured progress block for read_cell polling."""
        elapsed = time.time() - self.start_time
        
        # Calculate percentage
        base_pct = (self.current_category - 1) / self.total_categories * 100
        if self.current_phase > 0:
            base_pct += (self.current_phase / 5) / self.total_categories * 100
        
        cat_elapsed = time.time() - self.category_start if self.category_start else 0
        
        print()
        print("=" * 70)
        print(f"PROGRESS: {self.current_category}/{self.total_categories} categories | {base_pct:.0f}% | ETA: {self.estimate_remaining()}")
        print("-" * 70)
        print(f"Category: {self.current_category_name} ({self.current_category}/{self.total_categories})")
        if self.current_phase > 0:
            print(f"  Phase: [{self.current_phase}/5] {self.current_phase_name}")
        if self.current_detail:
            print(f"  Detail: {self.current_detail}")
        print(f"  Timing: {self._format_time(elapsed)} total | {self._format_time(cat_elapsed)} this category")
        print("=" * 70)
    
    def _update_progress_file(self):
        """Write progress to preset-specific file with accumulated statistics."""
        elapsed = time.time() - self.start_time
        
        # Merge with existing statistics
        merged_stats = self._merge_statistics()
        
        # Determine if this run completed (for total_runs counter)
        run_complete = self.current_category == self.total_categories and len(self.category_times) == self.total_categories
        total_runs = self.existing_data.get("total_runs", 0)
        if run_complete:
            total_runs += 1
        
        progress = {
            "preset": self.mode,
            "last_run": datetime.now().isoformat(),
            "total_runs": total_runs,
            "current_run": {
                "timestamp": datetime.now().isoformat(),
                "run": {
                    "elapsed_seconds": round(elapsed, 2),
                    "elapsed_formatted": self._format_time(elapsed),
                    "started": datetime.fromtimestamp(self.start_time).isoformat(),
                },
                "categories": {
                    "current": self.current_category,
                    "total": self.total_categories,
                    "current_name": self.current_category_name,
                    "completed": len(self.category_times),
                    "remaining": self.total_categories - self.current_category,
                },
                "phase": {
                    "current": self.current_phase,
                    "total": 5,
                    "name": self.current_phase_name,
                    "elapsed_seconds": round(time.time() - self.phase_start, 2) if self.phase_start else 0,
                },
                "detail": self.current_detail,
                "estimate": {
                    "remaining": self.estimate_remaining(),
                    "avg_category_seconds": round(sum(self.category_times) / len(self.category_times), 2) if self.category_times else None,
                },
            },
            "statistics": merged_stats,
            "category_breakdown": {
                cat: {phase: round(t, 2) for phase, t in phases.items()}
                for cat, phases in self.category_phase_history.items()
            },
        }
        
        try:
            with open(self.output_dir / f"progress_{self.mode}.yaml", "w") as f:
                yaml.dump(progress, f, default_flow_style=False, sort_keys=False)
        except Exception:
            pass

# =============================================================================
# Incremental Save Functions
# =============================================================================

def should_process_category(category: str, output_dir: Path, force: bool = False) -> bool:
    """Check if category needs processing (data preservation)."""
    status_file = output_dir / category / "_status.yaml"
    
    if force:
        return True
    
    if not status_file.exists():
        return True
    
    try:
        with open(status_file) as f:
            status = yaml.safe_load(f)
        if status.get("status") == "complete":
            return False
    except Exception:
        return True
    
    return True

def mark_category_status(output_dir: Path, category: str, status: str, error: str = None):
    """Update category processing status."""
    cat_dir = output_dir / category
    cat_dir.mkdir(exist_ok=True)
    
    data = {
        "status": status,
        "timestamp": datetime.now().isoformat(),
    }
    if error:
        data["error"] = error
    
    with open(cat_dir / "_status.yaml", "w") as f:
        yaml.dump(data, f)

def save_category_vectors(output_dir: Path, category: str, vectors: Dict[str, SteeringVector]):
    """Save vectors immediately after training."""
    cat_dir = output_dir / category
    cat_dir.mkdir(exist_ok=True)
    
    for name, vec in vectors.items():
        if vec is not None:
            vec.save(str(cat_dir / name))
    
    print(f"      [SAVED] {category}/*.svec ({len(vectors)} vectors)")

def save_category_phase(output_dir: Path, category: str, phase: str, data: dict):
    """Save a single phase's data immediately."""
    cat_dir = output_dir / category
    cat_dir.mkdir(exist_ok=True)
    
    filename_map = {
        "layer_stability": "layer_stability.yaml",
        "condition_search": "condition_search.yaml",
        "range_optimization": "range_optimization.yaml",
    }
    
    filename = filename_map.get(phase)
    if filename:
        with open(cat_dir / filename, "w") as f:
            yaml.dump(data, f, default_flow_style=False, sort_keys=False)
        print(f"      [SAVED] {category}/{filename}")

def update_master_config(output_dir: Path, category: str, result: CategoryResult, 
                         model_id: str, num_layers: int, hidden_size: int, mode: str):
    """Incrementally update config.yaml with completed category.
    
    Handles both new per-category format and legacy single-config format.
    """
    config_path = output_dir / "config.yaml"
    
    if config_path.exists():
        with open(config_path) as f:
            existing = yaml.safe_load(f) or {}
        
        # Check if it's the new per-category format
        if "categories" in existing:
            config = existing
        else:
            # Legacy format - start fresh but preserve for reference
            config = {
                "model": model_id,
                "num_layers": num_layers,
                "hidden_size": hidden_size,
                "mode": mode,
                "created": datetime.now().isoformat(),
                "categories": {},
                "_legacy_config": existing,  # Preserve old config for reference
            }
    else:
        config = {
            "model": model_id,
            "num_layers": num_layers,
            "hidden_size": hidden_size,
            "mode": mode,
            "created": datetime.now().isoformat(),
            "categories": {},
        }
    
    # Now safely add category
    config["categories"][category] = {
        "stable_layers": result.stable_layers,
        "training_layers": result.training_layers,
        "condition": {
            "layer": result.condition_layer,
            "threshold": float(result.condition_threshold),
            "direction": result.condition_direction,
            "f1": float(result.condition_f1),
        },
        "range": {
            "optimal_forward": float(result.optimal_forward),
            "max_forward": float(result.max_forward),
            "max_reverse": float(result.max_reverse),
            "balanced": float(result.balanced_range),
        },
        "vectors": {
            "forward": f"{category}/forward.svec",
            "reverse": f"{category}/reverse.svec",
            "condition": f"{category}/condition.svec",
            "blended": f"{category}/blended.svec",
        },
        "timing": result.timing,
    }
    config["last_updated"] = datetime.now().isoformat()
    
    with open(config_path, "w") as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    print(f"      [SAVED] config.yaml (added {category})")

# =============================================================================
# Core Training Functions
# =============================================================================

def train_behavior_vectors(model, tokenizer, questions, refusal, compliance, layer_ids=None):
    """Train forward/reverse behavior vectors."""
    GlobalConfig.set_verbose(False)
    examples = [(q, q) for q in questions]
    
    fwd_ds = SteeringDataset(tokenizer=tokenizer, examples=examples,
                             suffixes=list(zip(refusal, compliance)))
    fwd = SteeringVector.train(model=model, tokenizer=tokenizer, steering_dataset=fwd_ds,
                               method="pca_pairwise", accumulate_last_x_tokens="suffix-only")
    
    rev_ds = SteeringDataset(tokenizer=tokenizer, examples=examples,
                             suffixes=list(zip(compliance, refusal)))
    rev = SteeringVector.train(model=model, tokenizer=tokenizer, steering_dataset=rev_ds,
                               method="pca_pairwise", accumulate_last_x_tokens="suffix-only")
    GlobalConfig.set_verbose(True)
    
    if layer_ids:
        fwd = SteeringVector(model_type=fwd.model_type,
                            directions={k: v for k, v in fwd.directions.items() if k in layer_ids},
                            explained_variances={k: v for k, v in fwd.explained_variances.items() if k in layer_ids})
        rev = SteeringVector(model_type=rev.model_type,
                            directions={k: v for k, v in rev.directions.items() if k in layer_ids},
                            explained_variances={k: v for k, v in rev.explained_variances.items() if k in layer_ids})
    return fwd, rev

def train_condition_vector(model, tokenizer, harmful, harmless):
    """Train condition vector for category detection."""
    GlobalConfig.set_verbose(False)
    n = min(len(harmful), len(harmless))
    ds = SteeringDataset(tokenizer=tokenizer, examples=list(zip(harmful[:n], harmless[:n])),
                         suffixes=None, disable_suffixes=True)
    vec = SteeringVector.train(model=model, tokenizer=tokenizer, steering_dataset=ds,
                               method="pca_pairwise", accumulate_last_x_tokens="all")
    GlobalConfig.set_verbose(True)
    return vec

def create_blended_vector(fwd, rev, blend=0.7):
    """Blend reverse and negative forward vectors."""
    dirs = {}
    for layer in fwd.directions:
        if layer in rev.directions:
            b = blend * rev.directions[layer] + (1-blend) * (-fwd.directions[layer])
            norm = np.linalg.norm(b)
            dirs[layer] = b / norm if norm > 0 else b
    return SteeringVector(model_type=fwd.model_type, directions=dirs,
                          explained_variances=fwd.explained_variances)

def analyze_layer_stability(model, tokenizer, vector, test_prompts, 
                           stability_cfg: dict, gen_cfg: dict,
                           save_responses: bool = False, progress: ProgressTracker = None):
    """Analyze layer stability with all configurable parameters.
    
    Args:
        stability_cfg: Must contain: layers, strengths, coherence_threshold, min_range_width
        gen_cfg: Must contain: max_new_tokens_stability
    """
    max_tokens = gen_cfg.get("max_new_tokens_stability", 60)
    gen_settings = {"max_new_tokens": max_tokens, "do_sample": False, "pad_token_id": tokenizer.eos_token_id}
    
    layers_to_test = sorted(vector.directions.keys()) if stability_cfg["layers"] == "all" else stability_cfg["layers"]
    strengths = sorted(stability_cfg["strengths"])
    threshold = stability_cfg["coherence_threshold"]
    min_range_width = stability_cfg.get("min_range_width", 1.5)
    
    total = len(layers_to_test) * len(strengths) * len(test_prompts)
    print(f"      {len(layers_to_test)} layers x {len(strengths)} strengths x {len(test_prompts)} prompts = {total} inferences")
    
    with suppress_output():
        malleable = MalleableModel(model, tokenizer)
    
    results = {}
    for i, layer in enumerate(layers_to_test):
        if progress:
            progress.update_detail(f"Layer {layer} ({i+1}/{len(layers_to_test)})")
        
        single_vec = SteeringVector(
            model_type=vector.model_type,
            directions={layer: vector.directions[layer]},
            explained_variances={layer: vector.explained_variances.get(layer, 1.0)}
        )
        
        data = LayerStabilityData(layer=layer)
        
        for strength in strengths:
            coherent_count = 0
            responses_at_strength = []
            
            with suppress_output():
                malleable.steer(behavior_vector=single_vec, behavior_layer_ids=[layer],
                               behavior_vector_strength=strength)
            
            for prompt in test_prompts:
                with suppress_output():
                    response = malleable.respond(prompt, settings=gen_settings)
                is_coh, metrics = is_coherent_text(response)
                if is_coh:
                    coherent_count += 1
                if save_responses:
                    responses_at_strength.append({"prompt": prompt, "response": response[:200],
                                                  "coherent": is_coh, "metrics": metrics})
            
            with suppress_output():
                malleable.reset_leash_to_default()
            
            data.coherence_by_strength[strength] = coherent_count / len(test_prompts)
            if save_responses:
                data.responses[strength] = responses_at_strength
        
        # Calculate max forward/reverse
        for s in sorted([x for x in strengths if x > 0]):
            if data.coherence_by_strength.get(s, 0) >= threshold:
                data.max_forward = s
            else:
                break
        for s in sorted([x for x in strengths if x < 0], reverse=True):
            if data.coherence_by_strength.get(s, 0) >= threshold:
                data.max_reverse = s
            else:
                break
        
        data.range_width = data.max_forward - data.max_reverse
        data.stable = data.range_width >= min_range_width  # Use config value
        results[layer] = data
        clear_gpu()
    
    return results

def select_stable_layers(stability: Dict[int, LayerStabilityData], 
                        min_width: float = 1.5, 
                        middle_layer: Union[int, str] = "auto",
                        num_layers: int = 36) -> List[int]:
    """Select stable layers sorted by proximity to middle layer.
    
    Args:
        stability: Dict of layer -> LayerStabilityData
        min_width: Minimum range width to be considered stable
        middle_layer: "auto" (num_layers // 2) or explicit int
        num_layers: Total layers in model (used when middle_layer="auto")
    """
    # Resolve middle layer
    if middle_layer == "auto":
        mid = num_layers // 2
    else:
        mid = int(middle_layer)
    
    stable = [d for d in stability.values() if d.range_width >= min_width]
    stable.sort(key=lambda d: abs(d.layer - mid))
    return [d.layer for d in stable]

print("[OK] Core functions + MCP progress system defined")

[OK] Core functions + MCP progress system defined


In [54]:
# Cell 7: Per-Category Analysis with MCP Progress

# =============================================================================
# Configuration
# =============================================================================
FORCE_REPROCESS = True  # Set True to reprocess completed categories

# =============================================================================
# Initialize
# =============================================================================
output_dir = NOTEBOOK_DIR / "steering_config"
output_dir.mkdir(exist_ok=True)

RUN_START = datetime.now()
results: Dict[str, CategoryResult] = {}

# Generation settings from config
gen_settings = {
    "max_new_tokens": GENERATION_CFG["max_new_tokens_test"],
    "do_sample": False,
    "pad_token_id": tokenizer.eos_token_id
}

# Initialize progress tracker with preset mode
progress = ProgressTracker(
    total_categories=len(CATEGORIES),
    output_dir=output_dir,
    mode=MODE
)

# =============================================================================
# Pre-run Parameter Summary & Inference Calculation
# =============================================================================
print("=" * 70)
print(f"PER-CATEGORY ANALYSIS ({MODE.upper()} MODE)")
print("=" * 70)

# Calculate inference counts per category
sample_harmful = harmful_by_category[CATEGORIES[0]]
harmful_limit = STABILITY_CFG.get("harmful_prompts", len(sample_harmful))
harmless_count = STABILITY_CFG["test_prompts"]
stability_prompts = harmful_limit + harmless_count

layers_count = NUM_LAYERS if STABILITY_CFG["layers"] == "all" else len(STABILITY_CFG["layers"])
strengths_count = len(STABILITY_CFG["strengths"])

# Phase 2: Layer stability
phase2_inferences = layers_count * strengths_count * stability_prompts

# Phase 4: Condition search (approximate - depends on F1 optimization)
cond_layers = CONDITION_CFG["layer_range"][1] - CONDITION_CFG["layer_range"][0]
cond_thresholds = int((CONDITION_CFG["threshold_range"][1] - CONDITION_CFG["threshold_range"][0]) / CONDITION_CFG["threshold_step"])
phase4_inferences = cond_layers * cond_thresholds  # Approximate

# Phase 5: Range optimization
test_prompts_count = len(test_harmful_by_category[CATEGORIES[0]]) + len(test_harmless)
fwd_strengths = len(RANGE_CFG["forward_strengths"])
rev_strengths = len(RANGE_CFG["reverse_strengths"])
rev_test_prompts = RANGE_CFG.get("reverse_test_prompts", 4)
phase5_fwd = fwd_strengths * test_prompts_count
phase5_rev = rev_strengths * rev_test_prompts
phase5_inferences = phase5_fwd + phase5_rev

total_per_category = phase2_inferences + phase4_inferences + phase5_inferences

print()
print("PARAMETER SUMMARY")
print("-" * 70)
print(f"Categories: {len(CATEGORIES)} ({', '.join(CATEGORIES)})")
print()
print("Phase 2 - Layer Stability:")
print(f"  Layers to test: {layers_count} (config: {STABILITY_CFG['layers']})")
print(f"  Strengths: {strengths_count} values {STABILITY_CFG['strengths']}")
print(f"  Test prompts: {stability_prompts} ({harmful_limit} harmful + {harmless_count} harmless)")
print(f"  Max tokens: {GENERATION_CFG['max_new_tokens_stability']}")
print(f"  Coherence threshold: {STABILITY_CFG['coherence_threshold']}")
print(f"  → Inferences: {layers_count} × {strengths_count} × {stability_prompts} = {phase2_inferences}")
print()
print("Phase 4 - Condition Search:")
print(f"  Layer range: {CONDITION_CFG['layer_range']} ({cond_layers} layers)")
print(f"  Threshold range: {CONDITION_CFG['threshold_range']}, step {CONDITION_CFG['threshold_step']}")
print(f"  → Approximate inferences: ~{phase4_inferences}")
print()
print("Phase 5 - Range Optimization:")
print(f"  Forward strengths: {fwd_strengths} values {RANGE_CFG['forward_strengths']}")
print(f"  Reverse strengths: {rev_strengths} values {RANGE_CFG['reverse_strengths']}")
print(f"  Test prompts: {test_prompts_count} (forward), {rev_test_prompts} (reverse)")
print(f"  Max tokens: {GENERATION_CFG['max_new_tokens_test']}")
print(f"  → Inferences: ({fwd_strengths} × {test_prompts_count}) + ({rev_strengths} × {rev_test_prompts}) = {phase5_inferences}")
print()
print("-" * 70)
print(f"TOTAL per category: ~{total_per_category} inferences")
print(f"TOTAL all categories: ~{total_per_category * len(CATEGORIES)} inferences")
print("-" * 70)
print()
print(f"Output: {output_dir}")
print(f"Progress file: progress_{MODE}.yaml")
print(f"Force reprocess: {FORCE_REPROCESS}")

# Save run metadata at start
run_metadata = {
    "run_id": RUN_START.strftime("%Y-%m-%dT%H-%M-%S"),
    "mode": MODE,
    "model": MODEL_ID,
    "force_reprocess": FORCE_REPROCESS,
    "system": {
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1) if torch.cuda.is_available() else 0,
    },
    "config_used": {
        "stability": STABILITY_CFG,
        "condition": CONDITION_CFG,
        "training": TRAINING_CFG,
        "range_optimization": RANGE_CFG,
        "generation": GENERATION_CFG,
    },
    "inference_estimates": {
        "phase2_layer_stability": phase2_inferences,
        "phase4_condition_search": phase4_inferences,
        "phase5_range_optimization": phase5_inferences,
        "total_per_category": total_per_category,
        "total_all_categories": total_per_category * len(CATEGORIES),
    },
    "started": RUN_START.isoformat(),
    "status": "running",
}
with open(output_dir / "run_metadata.yaml", "w") as f:
    yaml.dump(run_metadata, f, default_flow_style=False, sort_keys=False)

# =============================================================================
# Main Loop
# =============================================================================
for cat in CATEGORIES:
    # Check if already complete (data preservation)
    if not should_process_category(cat, output_dir, FORCE_REPROCESS):
        progress.skip_category(cat)
        continue
    
    progress.start_category(cat)
    mark_category_status(output_dir, cat, "running")
    
    try:
        result = CategoryResult(category=cat)
        cat_harmful = harmful_by_category[cat]
        cat_test = get_test_prompts(cat)
        
        # ─────────────────────────────────────────────────────────────────────
        # Phase 1: Train behavior vectors
        # ─────────────────────────────────────────────────────────────────────
        progress.start_phase(1, "Training behavior vectors")
        t0 = time.time()
        
        with suppress_output():
            result.forward, result.reverse = train_behavior_vectors(
                model, tokenizer, questions, refusal_suffixes, compliance_suffixes
            )
        
        result.timing["behavior_training"] = time.time() - t0
        progress.complete_phase()
        
        # Save vectors immediately
        save_category_vectors(output_dir, cat, {"forward": result.forward, "reverse": result.reverse})
        print(f"      Done ({result.timing['behavior_training']:.1f}s)")
        
        # ─────────────────────────────────────────────────────────────────────
        # Phase 2: Layer stability analysis
        # ─────────────────────────────────────────────────────────────────────
        progress.start_phase(2, "Layer stability analysis")
        t0 = time.time()
        
        # Use harmful_prompts config to limit harmful prompts (speed optimization)
        harmful_limit = STABILITY_CFG.get("harmful_prompts", len(cat_harmful))
        test_prompts_stability = cat_harmful[:harmful_limit] + harmless_prompts[:STABILITY_CFG["test_prompts"]]
        
        result.layer_stability = analyze_layer_stability(
            model, tokenizer, result.forward, test_prompts_stability,
            stability_cfg=STABILITY_CFG,
            gen_cfg=GENERATION_CFG,
            save_responses=OUTPUT_CFG["save_responses"],
            progress=progress
        )
        
        # Use configurable parameters for stable layer selection
        result.stable_layers = select_stable_layers(
            result.layer_stability,
            min_width=STABILITY_CFG.get("min_range_width", 1.5),
            middle_layer=STABILITY_CFG.get("middle_layer", "auto"),
            num_layers=NUM_LAYERS
        )
        result.training_layers = result.stable_layers[:TRAINING_CFG["max_layers"]]
        result.timing["layer_stability"] = time.time() - t0
        progress.complete_phase()
        
        # Save layer stability data immediately
        layer_stability_data = {
            "category": cat,
            "config": STABILITY_CFG,
            "summary": {
                "total_layers": len(result.layer_stability),
                "stable_count": len(result.stable_layers),
                "stable_layers": result.stable_layers,
                "training_layers": result.training_layers,
            },
            "layers": {
                layer: {
                    "max_forward": float(d.max_forward),
                    "max_reverse": float(d.max_reverse),
                    "range_width": float(d.range_width),
                    "stable": d.stable,
                    "coherence_by_strength": {float(k): float(v) for k, v in d.coherence_by_strength.items()},
                }
                for layer, d in result.layer_stability.items()
            },
        }
        save_category_phase(output_dir, cat, "layer_stability", layer_stability_data)
        print(f"      {len(result.stable_layers)} stable, using {result.training_layers} ({result.timing['layer_stability']:.1f}s)")
        
        # ─────────────────────────────────────────────────────────────────────
        # Phase 3: Train condition vector
        # ─────────────────────────────────────────────────────────────────────
        progress.start_phase(3, "Training condition vector")
        t0 = time.time()
        
        with suppress_output():
            result.condition = train_condition_vector(model, tokenizer, cat_harmful, harmless_prompts)
        
        result.timing["condition_training"] = time.time() - t0
        progress.complete_phase()
        
        # Save condition vector immediately
        save_category_vectors(output_dir, cat, {"condition": result.condition})
        print(f"      Done ({result.timing['condition_training']:.1f}s)")
        
        # ─────────────────────────────────────────────────────────────────────
        # Phase 4: Find condition point
        # ─────────────────────────────────────────────────────────────────────
        progress.start_phase(4, "Finding condition point")
        progress.update_detail(f"Layers {CONDITION_CFG['layer_range']}, step {CONDITION_CFG['threshold_step']}")
        t0 = time.time()
        
        with suppress_output():
            malleable = MalleableModel(model, tokenizer)
            best_layer, best_thresh, best_dir, f1 = malleable.find_best_condition_point(
                positive_strings=cat_harmful,
                negative_strings=harmless_prompts,
                condition_vector=result.condition,
                layer_range=tuple(CONDITION_CFG["layer_range"]),
                max_layers_to_combine=1,
                threshold_range=tuple(CONDITION_CFG["threshold_range"]),
                threshold_step=CONDITION_CFG["threshold_step"],
            )
        
        result.condition_layer = best_layer if isinstance(best_layer, list) else [best_layer]
        result.condition_threshold = best_thresh
        result.condition_direction = best_dir
        result.condition_f1 = f1
        result.timing["condition_search"] = time.time() - t0
        progress.complete_phase()
        
        # Save condition search results
        condition_data = {
            "category": cat,
            "config": CONDITION_CFG,
            "best": {
                "layer": result.condition_layer,
                "threshold": float(result.condition_threshold),
                "direction": result.condition_direction,
                "f1_score": float(result.condition_f1),
            },
        }
        save_category_phase(output_dir, cat, "condition_search", condition_data)
        print(f"      L{result.condition_layer}@{best_thresh:.2f} F1={f1:.2f} ({result.timing['condition_search']:.1f}s)")
        
        # ─────────────────────────────────────────────────────────────────────
        # Phase 5: Range optimization (all from config)
        # ─────────────────────────────────────────────────────────────────────
        progress.start_phase(5, "Optimizing strength range")
        t0 = time.time()
        range_data = {"forward_tests": {}, "reverse_tests": {}}
        
        # Get config values
        forward_strengths = RANGE_CFG["forward_strengths"]
        reverse_strengths = RANGE_CFG["reverse_strengths"]
        reverse_test_prompts = RANGE_CFG.get("reverse_test_prompts", 4)
        accuracy_threshold = RANGE_CFG.get("accuracy_threshold", 0.75)
        min_optimal_forward = RANGE_CFG.get("min_optimal_forward", 0.5)
        
        # Test forward strengths
        max_fwd = 0.0
        for i, s in enumerate(forward_strengths):
            progress.update_detail(f"Forward strength {s} ({i+1}/{len(forward_strengths)})")
            
            with suppress_output():
                malleable.steer(
                    behavior_vector=result.forward,
                    behavior_layer_ids=result.training_layers,
                    behavior_vector_strength=s,
                    condition_vector=result.condition,
                    condition_layer_ids=result.condition_layer,
                    condition_vector_threshold=result.condition_threshold,
                    condition_comparator_threshold_is=result.condition_direction
                )
            
            correct = 0
            all_coherent = True
            for prompt, expected in cat_test:
                with suppress_output():
                    response = malleable.respond(prompt, settings=gen_settings)
                is_coh, _ = is_coherent_text(response)
                is_ref = is_refusal(response)
                exp_correct = (is_ref if expected == "harmful" else not is_ref)
                if is_coh and exp_correct:
                    correct += 1
                if not is_coh:
                    all_coherent = False
            
            with suppress_output():
                malleable.reset_leash_to_default()
            
            accuracy = correct / len(cat_test)
            range_data["forward_tests"][s] = {
                "accuracy": float(accuracy), "correct": correct, "total": len(cat_test), "coherent": all_coherent
            }
            if all_coherent and accuracy >= accuracy_threshold:
                max_fwd = s
        
        # Test reverse strengths (from config)
        progress.update_detail("Testing reverse strengths")
        max_rev = 0.0
        for s in reverse_strengths:
            with suppress_output():
                malleable.steer(
                    behavior_vector=result.forward,
                    behavior_layer_ids=result.training_layers,
                    behavior_vector_strength=s
                )
            
            all_coherent = True
            # Use configurable number of test prompts
            for prompt, _ in cat_test[:reverse_test_prompts]:
                with suppress_output():
                    response = malleable.respond(prompt, settings=gen_settings)
                is_coh, _ = is_coherent_text(response)
                if not is_coh:
                    all_coherent = False
                    break
            
            with suppress_output():
                malleable.reset_leash_to_default()
            
            range_data["reverse_tests"][s] = {"coherent": all_coherent}
            if all_coherent:
                max_rev = s
        
        result.range_data = range_data
        result.max_forward = max_fwd
        result.max_reverse = max_rev
        result.optimal_forward = max(min_optimal_forward, max_fwd / 2)
        result.balanced_range = min(max_fwd, abs(max_rev)) if max_fwd > 0 and max_rev < 0 else min_optimal_forward
        result.timing["range_optimization"] = time.time() - t0
        progress.complete_phase()
        
        # Save range optimization results
        range_save_data = {
            "category": cat,
            "config": RANGE_CFG,
            "training_layers": result.training_layers,
            "results": {
                "optimal_forward": float(result.optimal_forward),
                "max_forward": float(result.max_forward),
                "max_reverse": float(result.max_reverse),
                "balanced_range": float(result.balanced_range),
            },
            "forward_tests": {float(k): v for k, v in range_data["forward_tests"].items()},
            "reverse_tests": {float(k): v for k, v in range_data["reverse_tests"].items()},
        }
        save_category_phase(output_dir, cat, "range_optimization", range_save_data)
        print(f"      fwd={max_fwd:.1f}, rev={max_rev:.1f}, balanced={result.balanced_range:.1f} ({result.timing['range_optimization']:.1f}s)")
        
        # ─────────────────────────────────────────────────────────────────────
        # Finalize category
        # ─────────────────────────────────────────────────────────────────────
        # Create and save blended vector
        result.blended = create_blended_vector(result.forward, result.reverse, TRAINING_CFG["blend_ratio"])
        save_category_vectors(output_dir, cat, {"blended": result.blended})
        
        # Calculate total time
        result.timing["total"] = sum(result.timing.values())
        
        # Mark complete and update master config
        mark_category_status(output_dir, cat, "complete")
        update_master_config(output_dir, cat, result, MODEL_ID, NUM_LAYERS, HIDDEN_SIZE, MODE)
        
        progress.complete_category()
        results[cat] = result
        clear_gpu()
        
    except Exception as e:
        mark_category_status(output_dir, cat, "error", str(e))
        print(f"      [ERROR] {cat}: {e}")
        raise

# =============================================================================
# Finalize
# =============================================================================
RUN_END = datetime.now()

# Update run metadata
run_metadata["completed"] = RUN_END.isoformat()
run_metadata["status"] = "complete"
run_metadata["total_seconds"] = (RUN_END - RUN_START).total_seconds()
run_metadata["per_category_seconds"] = {cat: r.timing.get("total", 0) for cat, r in results.items()}
with open(output_dir / "run_metadata.yaml", "w") as f:
    yaml.dump(run_metadata, f, default_flow_style=False, sort_keys=False)

print()
print("=" * 70)
print("ANALYSIS COMPLETE")
print(f"Total time: {progress._format_time((RUN_END - RUN_START).total_seconds())}")
print(f"Categories processed: {len(results)}/{len(CATEGORIES)}")
print("=" * 70)

Searching for best condition point        ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━        100%  0:00:00  0:00:00   


Search completed.


Best condition point found: Layers [29], Threshold 0.060, Direction 'larger', F1 Score 1.000


Resetting leash to default...


Steering...


...

      [SAVED] sexual/range_optimization.yaml
      fwd=1.5, rev=-3.0, balanced=1.5 (9.6s)


Saving SteeringVector to /workspace/Sync/AI/bazzite/bazzite-ai-notebooks/steering_config/sexual/blended.svec


SteeringVector saved successfully


      [SAVED] sexual/*.svec (1 vectors)
      [SAVED] config.yaml (added sexual)

PROGRESS: 6/6 categories | 100% | ETA: 0.0s
----------------------------------------------------------------------
Category: sexual (6/6)
  Phase: [5/5] Optimizing strength range
  Detail: Testing reverse strengths
  Timing: 21m 2s total | 3m 29s this category



ANALYSIS COMPLETE
Total time: 21m 2s
Categories processed: 6/6


In [55]:
# Cell 8: Verify Saved Data

print("=" * 70)
print("VERIFYING SAVED DATA")
print("=" * 70)

import os

def count_files(directory: Path, pattern: str = "*") -> int:
    return len(list(directory.glob(pattern)))

print(f"\nOutput directory: {output_dir}")
print()

# Check each category
for cat in CATEGORIES:
    cat_dir = output_dir / cat
    if cat_dir.exists():
        status_file = cat_dir / "_status.yaml"
        status = "unknown"
        if status_file.exists():
            with open(status_file) as f:
                status_data = yaml.safe_load(f)
                status = status_data.get("status", "unknown")
        
        yaml_count = count_files(cat_dir, "*.yaml")
        svec_count = count_files(cat_dir, "*.svec")
        print(f"  {cat}: {status} ({yaml_count} yaml, {svec_count} vectors)")
    else:
        print(f"  {cat}: not processed")

# Check master files
print()
print("Master files:")
for fname in ["config.yaml", "run_metadata.yaml", f"progress_{MODE}.yaml"]:
    fpath = output_dir / fname
    if fpath.exists():
        size = os.path.getsize(fpath)
        print(f"  {fname}: {size:,} bytes")
    else:
        print(f"  {fname}: missing")

# Show progress file contents summary
progress_file = output_dir / f"progress_{MODE}.yaml"
if progress_file.exists():
    print()
    print(f"Progress file ({MODE} mode):")
    with open(progress_file) as f:
        progress_data = yaml.safe_load(f)
    print(f"  Total runs: {progress_data.get('total_runs', 0)}")
    print(f"  Last run: {progress_data.get('last_run', 'N/A')}")
    stats = progress_data.get("statistics", {})
    if stats.get("phase_averages"):
        print(f"  Phase averages:")
        for phase, avg in stats["phase_averages"].items():
            print(f"    {phase}: {avg:.1f}s")

print()
print("[OK] Data verification complete")

VERIFYING SAVED DATA

Output directory: /workspace/Sync/AI/bazzite/bazzite-ai-notebooks/steering_config

  violence: complete (4 yaml, 4 vectors)
  cybercrime: complete (4 yaml, 4 vectors)
  fraud: complete (4 yaml, 4 vectors)
  drugs: complete (4 yaml, 4 vectors)
  hate: complete (4 yaml, 4 vectors)
  sexual: complete (4 yaml, 4 vectors)

Master files:
  config.yaml: 6,329 bytes
  run_metadata.yaml: 1,308 bytes
  progress_quick.yaml: 3,022 bytes

Progress file (quick mode):
  Total runs: 1
  Last run: 2025-12-08T11:33:34.805514
  Phase averages:
    behavior_vectors: 25.4s
    layer_stability: 172.1s
    condition_vector: 0.3s
    condition_search: 2.5s
    range_optimization: 9.6s

[OK] Data verification complete


In [56]:
# Cell 9: Summary + Cleanup

print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"\nModel: {MODEL_ID}")
print(f"Mode: {MODE}")
if 'RUN_END' in dir() and 'RUN_START' in dir():
    total_time = (RUN_END - RUN_START).total_seconds()
    print(f"Total time: {total_time / 60:.1f} minutes ({total_time:.0f}s)")
print()

if results:
    print("Per-Category Results:")
    print(f"{'Category':<12} {'Stable':<8} {'Training':<12} {'Condition':<16} {'Range':<12}")
    print("-" * 60)
    for cat, result in results.items():
        cond_str = f"L{result.condition_layer[0]}@{result.condition_threshold:.2f}"
        range_str = f"[{result.max_reverse:.1f}, {result.max_forward:.1f}]"
        print(f"{cat:<12} {len(result.stable_layers):<8} {str(result.training_layers):<12} {cond_str:<16} {range_str:<12}")
else:
    print("No categories processed (all already complete?)")
    print("Set FORCE_REPROCESS = True in Cell 7 to reprocess")

print(f"\nOutput directory: {output_dir}")
print(f"  config.yaml         - Master config for 10b-steering-apply.ipynb")
print(f"  run_metadata.yaml   - Run timing and system info")
print(f"  progress_{MODE}.yaml - Progress stats (for {MODE} mode)")
print(f"  <category>/         - Per-category data and vectors")

print(f"\nNext steps:")
print(f"  1. Review: cat {output_dir}/config.yaml")
print(f"  2. Apply:  Open 10b-steering-apply.ipynb")

# Cleanup
clear_gpu()
print_gpu("Final")
print("\n[OK] Notebook complete!")

SUMMARY

Model: mistralai/Ministral-8B-Instruct-2410
Mode: quick
Total time: 21.0 minutes (1263s)

Per-Category Results:
Category     Stable   Training     Condition        Range       
------------------------------------------------------------
violence     36       [18, 17, 19, 16] L18@0.03         [-3.0, 3.0] 
cybercrime   36       [18, 17, 19, 16] L18@0.03         [-3.0, 3.0] 
fraud        35       [18, 17, 19, 16] L29@0.06         [-3.0, 3.0] 
drugs        36       [18, 17, 19, 16] L19@0.09         [-3.0, 3.0] 
hate         36       [18, 17, 19, 16] L22@0.09         [-3.0, 1.5] 
sexual       36       [18, 17, 19, 16] L29@0.06         [-3.0, 1.5] 

Output directory: /workspace/Sync/AI/bazzite/bazzite-ai-notebooks/steering_config
  config.yaml      - Master config for 10b-steering-apply.ipynb
  run_metadata.yaml - Run timing and system info
  _progress.yaml   - Live progress (for MCP monitoring)
  <category>/      - Per-category data and vectors

Next steps:
  1. Review: cat /works